# 🚀 SpaceX Falcon 9 Landing Prediction
## Notebook 2 — Web Scraping Launch Records from Wikipedia

Wikipedia maintains a comprehensive table of every Falcon 9 and Falcon Heavy launch.
We use `requests` + `BeautifulSoup` to scrape it.

**Important:** Wikipedia blocks requests that don't send a browser-like `User-Agent`.
We fix this with a single header — no extra libraries needed.


In [27]:
%pip install pandas requests beautifulsoup4 lxml -q
import requests
from bs4 import BeautifulSoup
import unicodedata
import re
import pandas as pd
import numpy as np


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Helper functions

In [28]:
def clean_text(cell):
    """Strip footnotes, whitespace and unicode oddities from a table cell."""
    text = unicodedata.normalize("NFKD", cell.get_text())
    text = re.sub(r'\[\d+\]', '', text)   # remove [1], [2] ... footnotes
    return text.strip()

def extract_date(cell):
    strings = [s.strip() for s in cell.strings if s.strip()]
    return strings[0] if strings else np.nan

def extract_booster(cell):
    anchors = [a.get_text(strip=True) for a in cell.find_all('a')]
    return ' '.join(anchors) if anchors else clean_text(cell)

def parse_mass(val):
    """Extract numeric kg value from strings like '3,500 kg (7,700 lb)'."""
    m = re.search(r'([\d,]+)\s*kg', str(val))
    return float(m.group(1).replace(',', '')) if m else np.nan


### Fetch Wikipedia page

A browser-style `User-Agent` header is required — Wikipedia returns **403 Forbidden**
for plain `requests.get()` calls without it.


In [29]:
WIKI_URL = (
    "https://en.wikipedia.org/wiki/"
    "List_of_Falcon_9_and_Falcon_Heavy_launches"
)

# Mimic a real browser — required to avoid HTTP 403
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

response = requests.get(WIKI_URL, headers=HEADERS, timeout=20)
print(f"HTTP status: {response.status_code}")
assert response.status_code == 200, "Failed to fetch page — check your internet connection."

soup = BeautifulSoup(response.text, "html.parser")


HTTP status: 200


### Locate launch tables

In [30]:
# Wikipedia uses 'wikitable sortable' for all launch record tables
all_tables = soup.find_all("table", {"class": lambda c: c and "wikitable" in c})
print(f"Total wikitables found: {len(all_tables)}")

# Find tables that look like launch records (have 'Flight' or 'No.' in the header)
launch_tables = []
for t in all_tables:
    rows = t.find_all("tr")
    if not rows:
        continue
    header_text = rows[0].get_text()
    if any(kw in header_text for kw in ['Flight', 'Booster', 'Payload', 'No.']):
        launch_tables.append(t)

print(f"Launch tables identified: {len(launch_tables)}")


Total wikitables found: 4
Launch tables identified: 4


### Inspect column headers of the first launch table

In [31]:
header_row = launch_tables[0].find("tr")
cols = [clean_text(th) for th in header_row.find_all("th")]
print("Columns detected:", cols)


Columns detected: ['Flight No.', 'Date andtime (UTC)', 'Version,booster[j]', 'Launchsite', 'Payload[k]', 'Payload mass', 'Orbit', 'Customer', 'Launchoutcome', 'Boosterlanding']


### Parse all rows across all launch tables

In [32]:
all_rows = []

for table in launch_tables:

    rows = table.find_all("tr")[1:]

    for row in rows:

        cells = row.find_all(["td", "th"])

        if len(cells) < 10:
            continue

        try:
            all_rows.append({
                "Date": extract_date(cells[1]),
                "BoosterVersion": clean_text(cells[2]),
                "LaunchSite": clean_text(cells[3]),
                "Payload": clean_text(cells[4]),
                "PayloadMass_kg": parse_mass(clean_text(cells[5])),
                "Orbit": clean_text(cells[6]),
                "Outcome": clean_text(cells[8]),
            })

        except Exception:
            continue

wiki_df = pd.DataFrame(all_rows)

print(f"Rows scraped: {len(wiki_df)}")
wiki_df.head()

Rows scraped: 236


,Date,BoosterVersion,LaunchSite,Payload,PayloadMass_kg,Orbit,Outcome
0,"January 4, 2025",F9 B5B1073‐20,"Cape Canaveral, SLC‐40",Thuraya 4-NGS,5000.0,GTO,Success
1,"January 6, 2025",F9 B5B1077‐17,"Cape Canaveral, SLC‐40",Starlink: Group 6‐71,17500.0,LEO,Success
2,"January 8, 2025",F9 B5B1086‐3,"Kennedy, LC‐39A",Starlink: Group 12-11 (21 satellites),16500.0,LEO,Success
3,"January 10, 2025",F9 B5B1071‐22,"Vandenberg, SLC‐4E",NROL-153 (22 Starshield satellites),NaN,LEO,Success
4,"January 10, 2025",F9 B5B1067‐25,"Cape Canaveral, SLC‐40",Starlink: Group 12-12 (21 satellites),16500.0,LEO,Success


### Quick sanity checks

In [33]:
print("Missing PayloadMass:", wiki_df['PayloadMass_kg'].isnull().sum())
print()
print("Launch Sites:\n", wiki_df['LaunchSite'].value_counts().head(6))
print()
print("Top Orbits:\n", wiki_df['Orbit'].value_counts().head(8))


Missing PayloadMass: 18

Launch Sites:
 LaunchSite
Cape Canaveral, SLC‐40    108
Vandenberg, SLC‐4E         98
Kennedy, LC‐39A            28
Vandenberg, SLC-4E          2
Name: count, dtype: int64

Top Orbits:
 Orbit
LEO                   155
SSO                    53
GTO                     9
LEO (ISS)               9
MEO                     4
TLI                     2
Polar LEO               2
Polar (Retrograde)      1
Name: count, dtype: int64


### Save

In [34]:
wiki_df.to_csv("../data/spacex_launches_wiki.csv", index=False)
print("Saved → data/spacex_launches_wiki.csv")


Saved → data/spacex_launches_wiki.csv
